# 03 Robustness checks

**Data:** chosen in `common.py` (`USE_FAKE_DATA`). Chart titles say whether it is real or fake.

Do the conclusions still hold when reasonable choices change?

1. Rising threshold: 1.1, 1.2, 1.3
2. Period: full test, 2025 only, without spike weeks
3. Top-K for buyers: 5, 10, 20 styles per week
4. Random seeds (only when the file has a `seed` column)

Each table ends with **beats best baseline?**: does the main model score higher than the best baseline in that setting?

In [1]:
import pandas as pd

from common import load_predictions, load_seed_runs, spike_weeks, RISING_THRESHOLD, MAIN_MODEL, BASELINES

MODEL = MAIN_MODEL
T = RISING_THRESHOLD  # the team's chosen threshold
HORIZON = 4  # change to 8 to rerun every check for 8 weeks ahead

df = load_predictions()
df = df[(df["horizon"] == HORIZON) & df["model"].isin([MODEL] + BASELINES)].copy()

# Spike weeks flagged by Person 1 (spike_weeks.csv)
SPIKE_WEEKS = spike_weeks()

# A forecast "touches the spike" if any of the 4 weeks it predicts is a spike week.
def touches_spike(row):
    window_end = row["origin"] + pd.Timedelta(weeks=int(row["horizon"]))
    window_start = window_end - pd.Timedelta(weeks=3)
    return any(window_start <= week <= window_end for week in SPIKE_WEEKS)

df["touches_spike"] = df.apply(touches_spike, axis=1)
print(f"{len(df):,} forecasts; {df['touches_spike'].mean():.0%} of them touch spike weeks")

C:\Users\tisab\AppData\Local\Temp\ipykernel_24448\4187009847.py:17: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  window_end = row["origin"] + pd.Timedelta(weeks=int(row["horizon"]))
C:\Users\tisab\AppData\Local\Temp\ipykernel_24448\4187009847.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  window_start = window_end - pd.Timedelta(weeks=3)


35,109 forecasts; 18% of them touch spike weeks


## Shared scoring

Precision and recall as in notebook 01, for any threshold.

In [2]:
def precision_recall(group, threshold):
    said = group["predicted_growth"] > threshold
    rose = group["actual_growth"] > threshold
    correct = (said & rose).sum()
    precision = 100 * correct / said.sum() if said.sum() else float("nan")
    recall = 100 * correct / rose.sum() if rose.sum() else float("nan")
    return round(precision, 1), round(recall, 1)


def compare(data, threshold):
    """One row: main model and best baseline, with the yes/no verdict."""
    p_main, r_main = precision_recall(data[data["model"] == MODEL], threshold)
    baseline_scores = {b: precision_recall(data[data["model"] == b], threshold) for b in BASELINES}
    best = max(baseline_scores, key=lambda b: baseline_scores[b][0] + baseline_scores[b][1])
    p_base, r_base = baseline_scores[best]
    return {
        f"{MODEL} precision": p_main, f"{MODEL} recall": r_main,
        "best baseline": best, "baseline precision": p_base, "baseline recall": r_base,
        "beats best baseline?": "yes" if (p_main >= p_base and r_main >= r_base) else "no",
    }

## 1. Different thresholds

"Beats" means at least as good on **both** precision and recall.

In [3]:
rows = {f"threshold {t}": compare(df, t) for t in (1.1, 1.2, 1.3)}
pd.DataFrame(rows).T

,lightgbm precision,lightgbm recall,best baseline,baseline precision,baseline recall,beats best baseline?
threshold 1.1,57.8,47.8,last_value,69.5,38.3,no
threshold 1.2,43.3,29.7,last_value,61.3,24.6,no
threshold 1.3,31.3,18.0,last_value,58.0,20.0,no


## 2. Different periods (team threshold)

The proposal promised all three views, because the spring 2026 spike could make the test period look better or worse than normal.

In [4]:
periods = {
    "full test period": df,
    "2025 only": df[df["origin"].dt.year == 2025],
    "without spike weeks": df[~df["touches_spike"]],
}
pd.DataFrame({name: compare(data, T) for name, data in periods.items()}).T

,lightgbm precision,lightgbm recall,best baseline,baseline precision,baseline recall,beats best baseline?
full test period,43.3,29.7,last_value,61.3,24.6,no
2025 only,71.3,40.7,seasonal_naive,33.9,33.8,yes
without spike weeks,35.3,36.6,seasonal_naive,41.3,29.4,no


## 3. Top-K for buyers

Each week, take the K styles with the highest predicted growth. **Precision at K** = share of those K that actually rose (growth above the team threshold), averaged over all weeks.

This is how a buyer would use the app: stock up on the top K.

In [5]:
def precision_at_k(data, k, threshold=T):
    weekly = []
    for _, week in data.groupby("origin"):
        top = week.nlargest(k, "predicted_growth")
        weekly.append((top["actual_growth"] > threshold).mean())
    return round(100 * sum(weekly) / len(weekly), 1)


rows = {}
for k in (5, 10, 20):
    scores = {m: precision_at_k(df[df["model"] == m], k) for m in [MODEL] + BASELINES}
    best_baseline = max(scores[b] for b in BASELINES)
    rows[f"top {k}"] = {**{f"{m} (%)": s for m, s in scores.items()},
                        "beats best baseline?": "yes" if scores[MODEL] >= best_baseline else "no"}

# For comparison: picking K styles at random would give roughly the share of styles that rise.
print(f"Random picks would be right about {100 * (df[df['model'] == MODEL]['actual_growth'] > T).mean():.1f}% of the time")
pd.DataFrame(rows).T

Random picks would be right about 20.3% of the time


,lightgbm (%),last_value (%),seasonal_naive (%),beats best baseline?
top 5,61.1,46.1,41.3,yes
top 10,55.3,43.0,37.2,yes
top 20,44.0,38.1,32.8,yes


## 4. Random seeds

Tree models give slightly different results each time they are trained. If the file has a `seed` column (5 runs), this shows the spread of precision and recall across runs.

In [6]:
full = load_seed_runs()
if "seed" not in full.columns:
    print("No `seed` column in this file yet: this check runs once Person 2's file includes it.")
else:
    runs = full[full["horizon"] == HORIZON]
    per_seed = pd.DataFrame(
        {seed: dict(zip(["precision", "recall"], precision_recall(group, T))) for seed, group in runs.groupby(runs["seed"].astype(int))}
    ).T
    print(per_seed)
    print("Spread (max - min):", (per_seed.max() - per_seed.min()).round(1).to_dict())

   precision  recall
0       43.8    28.8
1       42.2    28.9
2       43.1    30.4
3       42.8    29.4
4       44.0    29.9
Spread (max - min): {'precision': 1.8, 'recall': 1.6}


## What this shows (will be shown once real results are in)

1. Thresholds: the model beats the best baseline in __ of 3 settings.
2. Periods: results with and without spike weeks differ by __ points, so the spike __.
3. Top-K: a buyer stocking the top 10 would be right __% of the time vs. __% for the best baseline and __% for random picks.
4. Seeds: precision varies by __ points across runs.